<a href="https://colab.research.google.com/github/pkdev0101/Flyrank-ML-internship-Pranay-Kamath/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pkdev0101/Flyrank-ML-internship-Pranay-Kamath/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh / Content Opportunity Scoring. I will frame it mainly as a ranking and scoring task. Each content page will receive a review-priority score, and the pages will be sorted from strongest to weakest candidate. The output is not an automatic command to change a page. It is a decision-support queue that helps a content editor decide which pages to inspect first and whether to refresh, expand, protect, prune, or monitor them.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For this starter version, my proxy target will be decline_proxy. It will equal 1 when trend_direction is "down" and 0 otherwise. This is a defined proxy from the current data window, not a perfect future observed outcome.

I will not use trend_direction or trend_pct as model features because they directly define or reveal the proxy target. That would cause leakage. A stronger future version of the project would use signals from a prior 90-day window to predict decline or recovery during a later 30-day window.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

My main success metric will be Precision@50. It measures how many of the first 50 pages in the ranked review queue match the chosen proxy target.

This metric fits the real action because an editor has limited time and may review only the first 50 recommendations. I will compare my method with the starter rule baseline, which achieved Precision@50 of 0.240. A useful result should beat 0.240 using client-holdout validation. My initial practical target is at least 0.500, meaning at least 25 of the first 50 recommended pages match the proxy.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [3]:
import os
import sys
import subprocess

import pandas as pd

REPO_URL = "https://github.com/pkdev0101/Flyrank-ML-internship-Pranay-Kamath"
REPO_DIR = "/content/Flyrank-ML-internship-Pranay-Kamath"

# Open the repository when running in Google Colab.
if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Use the same basic eligibility rules as the starter pipeline.
lane_df = (
    df.loc[
        (df["impressions_90d"] > 0)
        & (df["content_age_days"] >= 90),
        [
            "content_id",
            "client_id",
            "content_age_days",
            "days_since_last_update",
            "impressions_90d",
            "clicks_90d",
            "sessions_90d",
            "avg_position",
            "ctr",
            "engagement_rate",
            "scroll_rate",
            "word_count",
            "trend_direction",
        ],
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

# Starter proxy target.
lane_df["decline_proxy"] = (
    lane_df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

# Do not display trend_direction as a proposed model feature.
display_columns = [
    "content_id",
    "client_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "decline_proxy",
]

print("Unit of analysis: one row = one pseudonymized content page")
print(f"Eligible pages: {len(lane_df):,}")
print(f"Duplicate content IDs: {lane_df['content_id'].duplicated().sum():,}")
print(f"Decline proxy rate: {lane_df['decline_proxy'].mean():.1%}")

lane_df[display_columns].head(10)

Unit of analysis: one row = one pseudonymized content page
Eligible pages: 30,000
Duplicate content IDs: 0
Decline proxy rate: 54.2%


,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,word_count,decline_proxy
0,content_304f48230142,client_f369cb89fc,187,20,3803,29,17,10.6,0.76,5.88,4.55,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,445,25,15320,7,9,20.3,0.05,0.00,10.00,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,141,20,12581,11,11,36.5,0.09,0.00,28.57,3515.0,1
3,content_331d6c4de07b,client_19581e27de,463,22,11751,58,78,6.2,0.49,1.28,3.45,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,263,14,19140,24,145,44.0,0.13,0.00,24.29,2803.0,1
5,content_d4084a4bc775,client_f369cb89fc,147,20,3970,1,5,8.5,0.03,0.00,25.00,3080.0,1
6,content_9a34b442b552,client_8722616204,90,20,20,0,1,7.0,0.00,0.00,0.00,3059.0,1
7,content_a63219c6e95a,client_19581e27de,445,22,1724,1,28,21.2,0.06,3.57,7.14,NaN,0
8,content_5e6c160719bc,client_6208ef0f77,90,20,32574,29,68,46.0,0.09,5.88,6.25,3807.0,1
9,content_c27558df2b0c,client_19581e27de,257,104,1240,2,3,4.9,0.16,0.00,0.00,NaN,1


The unit of analysis is one pseudonymized content page. Each row contains observable page-level search, freshness, content, and engagement measurements. The decline_proxy column sketches the starter target. The content_id and client_id columns identify and group rows, but they will not be model features.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can be useful as a baseline, but the pattern is more complicated than one if-statement. In my earlier data check, only 17 pages met the strict stale-and-visible rule, while 9,745 visible pages met the low-CTR condition. This shows that one rule can be too narrow while another can produce far too many candidates.

A scoring model can combine impressions, position, CTR, page age, freshness, sessions, engagement, and content depth. It may learn interactions such as when low CTR matters because a page has enough impressions and a strong enough search position. ML earns its place only if it beats a transparent baseline using honest validation and still gives understandable reason codes. A human reviewer will make the final decision.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.